# 03 - Temporal Analysis of COVID-19 Vaccination

## Objective
Analyze how COVID-19 vaccination changed over time in India and across states from January to August 2021.

### Questions
1. How did cumulative vaccination grow?
2. How did first and second doses change?
3. What were the daily, weekly, and monthly vaccination trends?
4. What were the major vaccination peaks?
5. How did vaccination differ across states over time?
6. How did vaccine-type and age-wise reporting change?
7. How did vaccination progress around the second COVID-19 wave?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

vaccination = pd.read_csv("../data/vaccination_clean.csv")
vaccination["Updated On"] = pd.to_datetime(vaccination["Updated On"])

print("Shape:", vaccination.shape)
print("Date range:", vaccination["Updated On"].min(), "to", vaccination["Updated On"].max())
print("States:", vaccination["State"].nunique())


FileNotFoundError: [Errno 2] No such file or directory: 'vaccination_clean.csv'

## 1. Prepare national and state-level data

India-level rows are used for national trends. State rows are kept separately so the national aggregate is not double-counted.


In [ ]:
india_vaccination = (
    vaccination[vaccination["State"] == "India"]
    .copy()
    .sort_values("Updated On")
    .reset_index(drop=True)
)

state_vaccination = (
    vaccination[vaccination["State"] != "India"]
    .copy()
    .sort_values(["State", "Updated On"])
    .reset_index(drop=True)
)

print("India records:", len(india_vaccination))
print("State records:", len(state_vaccination))


## 2. Derive daily vaccination measures

The original dose variables are cumulative, so first differences estimate changes between reporting dates. Negative changes are retained because they correspond to documented revisions.

In [ ]:
india_vaccination["New Doses"] = india_vaccination["Total Doses Administered"].diff()
india_vaccination["New First Doses"] = india_vaccination["First Dose Administered"].diff()
india_vaccination["New Second Doses"] = india_vaccination["Second Dose Administered"].diff()

india_vaccination["New Doses 7D MA"] = india_vaccination["New Doses"].rolling(7, min_periods=1).mean()
india_vaccination["New First Doses 7D MA"] = india_vaccination["New First Doses"].rolling(7, min_periods=1).mean()
india_vaccination["New Second Doses 7D MA"] = india_vaccination["New Second Doses"].rolling(7, min_periods=1).mean()

print(india_vaccination[[
    "Updated On","Total Doses Administered","New Doses",
    "First Dose Administered","New First Doses",
    "Second Dose Administered","New Second Doses"
]].head(10))


## 3. Cumulative vaccination trend

In [ ]:
fig, ax = plt.subplots()
ax.plot(india_vaccination["Updated On"], india_vaccination["Total Doses Administered"], label="Total doses")
ax.plot(india_vaccination["Updated On"], india_vaccination["First Dose Administered"], label="First dose")
ax.plot(india_vaccination["Updated On"], india_vaccination["Second Dose Administered"], label="Second dose")
ax.set_title("Cumulative COVID-19 Vaccination in India")
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative doses")
ax.legend()
plt.tight_layout()
plt.show()


## 4. Daily vaccination activity and 7-day moving average

In [ ]:
fig, ax = plt.subplots()
ax.plot(india_vaccination["Updated On"], india_vaccination["New Doses"], label="Daily new doses", alpha=0.5)
ax.plot(india_vaccination["Updated On"], india_vaccination["New Doses 7D MA"], label="7-day average")
ax.set_title("Daily and 7-Day Average Vaccination Activity")
ax.set_xlabel("Date")
ax.set_ylabel("New doses")
ax.legend()
plt.tight_layout()
plt.show()


## 5. First-dose vs second-dose activity

In [ ]:
fig, ax = plt.subplots()
ax.plot(india_vaccination["Updated On"], india_vaccination["New First Doses 7D MA"], label="First doses")
ax.plot(india_vaccination["Updated On"], india_vaccination["New Second Doses 7D MA"], label="Second doses")
ax.set_title("7-Day Average of New First and Second Doses")
ax.set_xlabel("Date")
ax.set_ylabel("Doses")
ax.legend()
plt.tight_layout()
plt.show()


## 6. Peak vaccination activity

In [ ]:
peak_idx = india_vaccination["New Doses 7D MA"].idxmax()
peak = india_vaccination.loc[peak_idx]

first_peak_idx = india_vaccination["New First Doses 7D MA"].idxmax()
second_peak_idx = india_vaccination["New Second Doses 7D MA"].idxmax()

print("Overall vaccination peak:", peak["Updated On"].date())
print("Peak 7-day average:", round(peak["New Doses 7D MA"]))
print("First-dose peak:", india_vaccination.loc[first_peak_idx, "Updated On"].date(),
      round(india_vaccination.loc[first_peak_idx, "New First Doses 7D MA"]))
print("Second-dose peak:", india_vaccination.loc[second_peak_idx, "Updated On"].date(),
      round(india_vaccination.loc[second_peak_idx, "New Second Doses 7D MA"]))


## 7. Weekly vaccination trend

In [ ]:
weekly = (
    india_vaccination.set_index("Updated On")["New Doses"]
    .resample("W").sum()
    .reset_index(name="Weekly New Doses")
)

fig, ax = plt.subplots()
ax.bar(weekly["Updated On"], weekly["Weekly New Doses"], width=5)
ax.set_title("Weekly COVID-19 Vaccine Doses Administered")
ax.set_xlabel("Week")
ax.set_ylabel("New doses")
plt.tight_layout()
plt.show()


## 8. Monthly vaccination trend

In [ ]:
monthly = (
    india_vaccination.set_index("Updated On")["New Doses"]
    .resample("MS").sum()
    .reset_index(name="Monthly New Doses")
)
monthly["Month"] = monthly["Updated On"].dt.strftime("%b %Y")
monthly["MoM Growth %"] = monthly["Monthly New Doses"].pct_change() * 100

print(monthly[["Month","Monthly New Doses","MoM Growth %"]])

fig, ax = plt.subplots()
ax.bar(monthly["Month"], monthly["Monthly New Doses"])
ax.set_title("Monthly COVID-19 Vaccine Doses Administered")
ax.set_xlabel("Month")
ax.set_ylabel("New doses")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 9. Cumulative vaccination milestones

In [ ]:
milestones = [10_000_000, 50_000_000, 100_000_000, 200_000_000, 300_000_000, 400_000_000]

for milestone in milestones:
    reached = india_vaccination[india_vaccination["Total Doses Administered"] >= milestone]
    if len(reached):
        print(f"{milestone:,} cumulative doses reached on {reached.iloc[0]['Updated On'].date()}")
    else:
        print(f"{milestone:,}: not reached in the available period")


## 10. State-level cumulative vaccination at latest reporting date

In [ ]:
latest_state = (
    state_vaccination.sort_values("Updated On")
    .groupby("State", as_index=False)
    .tail(1)
    .sort_values("Total Doses Administered", ascending=False)
)

print(latest_state[["State","Updated On","Total Doses Administered"]].head(15))

top_states = latest_state.head(10).sort_values("Total Doses Administered")

fig, ax = plt.subplots()
ax.barh(top_states["State"], top_states["Total Doses Administered"])
ax.set_title("Top States by Cumulative Vaccine Doses")
ax.set_xlabel("Cumulative doses")
ax.set_ylabel("State")
plt.tight_layout()
plt.show()


## 11. State-level vaccination summary

In [ ]:
state_summary = (
    state_vaccination.groupby("State")
    .agg(
        Start_Date=("Updated On","min"),
        End_Date=("Updated On","max"),
        Total_Doses=("Total Doses Administered","max"),
        First_Dose=("First Dose Administered","max"),
        Second_Dose=("Second Dose Administered","max")
    )
    .reset_index()
    .sort_values("Total_Doses", ascending=False)
)

print(state_summary.head(15))


## 12. First-dose and second-dose shares

In [ ]:
india_vaccination["First Dose Share %"] = (
    india_vaccination["First Dose Administered"] /
    india_vaccination["Total Doses Administered"] * 100
)
india_vaccination["Second Dose Share %"] = (
    india_vaccination["Second Dose Administered"] /
    india_vaccination["Total Doses Administered"] * 100
)

fig, ax = plt.subplots()
ax.plot(india_vaccination["Updated On"], india_vaccination["First Dose Share %"], label="First-dose share")
ax.plot(india_vaccination["Updated On"], india_vaccination["Second Dose Share %"], label="Second-dose share")
ax.set_title("First and Second Dose Shares of Cumulative Doses")
ax.set_xlabel("Date")
ax.set_ylabel("Share (%)")
ax.legend()
plt.tight_layout()
plt.show()


## 13. Vaccine-type reporting over time

Sputnik V begins reporting later in the dataset. Missing values are not imputed to zero.

In [ ]:
fig, ax = plt.subplots()
for col in ["Covaxin (Doses Administered)","CoviShield (Doses Administered)","Sputnik V (Doses Administered)"]:
    ax.plot(india_vaccination["Updated On"], india_vaccination[col], label=col)
ax.set_title("Cumulative Vaccine Doses by Vaccine Type")
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative doses")
ax.legend()
plt.tight_layout()
plt.show()


## 14. Age-wise dose reporting

Age-wise dose fields are available from 25 June 2021. This section therefore visualizes only reported observations.

In [ ]:
age_cols = [
    "18-44 Years (Doses Administered)",
    "45-60 Years (Doses Administered)",
    "60+ Years (Doses Administered)"
]

fig, ax = plt.subplots()
for col in age_cols:
    ax.plot(india_vaccination["Updated On"], india_vaccination[col], label=col)
ax.set_title("Reported Age-wise Cumulative Vaccine Doses")
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative doses")
ax.legend()
plt.tight_layout()
plt.show()


## 15. AEFI reporting over time

AEFI reporting begins on 16 March 2021. Missing values before reporting begins are preserved.

In [ ]:
aefi = india_vaccination.dropna(subset=["AEFI"])

if len(aefi):
    fig, ax = plt.subplots()
    ax.plot(aefi["Updated On"], aefi["AEFI"])
    ax.set_title("Reported AEFI Over Time")
    ax.set_xlabel("Date")
    ax.set_ylabel("AEFI")
    plt.tight_layout()
    plt.show()
    print("AEFI period:", aefi["Updated On"].min().date(), "to", aefi["Updated On"].max().date())


## 16. Vaccination around the second COVID-19 wave

The COVID analysis identified 9 May 2021 as the second-wave peak. This comparison is descriptive and does not establish causation.

In [ ]:
second_wave_peak = pd.Timestamp("2021-05-09")

window = india_vaccination[
    (india_vaccination["Updated On"] >= second_wave_peak - pd.Timedelta(days=45)) &
    (india_vaccination["Updated On"] <= second_wave_peak + pd.Timedelta(days=45))
]

fig, ax = plt.subplots()
ax.plot(window["Updated On"], window["New Doses 7D MA"])
ax.axvline(second_wave_peak, linestyle="--", label="Second-wave peak: 9 May 2021")
ax.set_title("Vaccination Activity Around the Second COVID-19 Wave Peak")
ax.set_xlabel("Date")
ax.set_ylabel("New doses - 7-day average")
ax.legend()
plt.tight_layout()
plt.show()

before = window[
    window["Updated On"] < second_wave_peak
]["New Doses 7D MA"].mean()
after = window[
    window["Updated On"] >= second_wave_peak
]["New Doses 7D MA"].mean()

print("Mean 7-day vaccination activity before peak:", round(before))
print("Mean 7-day vaccination activity from peak onward:", round(after))


## 17. Vaccination acceleration

In [ ]:
india_vaccination["Vaccination Acceleration"] = india_vaccination["New Doses 7D MA"].diff()

fig, ax = plt.subplots()
ax.plot(india_vaccination["Updated On"], india_vaccination["Vaccination Acceleration"])
ax.axhline(0, linestyle="--")
ax.set_title("Change in 7-Day Average Vaccination Activity")
ax.set_xlabel("Date")
ax.set_ylabel("Change in average daily doses")
plt.tight_layout()
plt.show()


## 18. Highest vaccination periods

In [ ]:
top_periods = (
    india_vaccination[["Updated On","New Doses 7D MA"]]
    .dropna()
    .sort_values("New Doses 7D MA", ascending=False)
    .head(10)
)
print(top_periods.to_string(index=False))


## 19. Temporal summary

In [ ]:
print("Study start:", india_vaccination["Updated On"].min().date())
print("Study end:", india_vaccination["Updated On"].max().date())
print("Initial cumulative doses:", int(india_vaccination["Total Doses Administered"].iloc[0]))
print("Final cumulative doses:", int(india_vaccination["Total Doses Administered"].iloc[-1]))
print("Peak 7-day average:", round(india_vaccination["New Doses 7D MA"].max()))
print("Peak date:", india_vaccination.loc[india_vaccination["New Doses 7D MA"].idxmax(),"Updated On"].date())


# Key findings to report

Use the numerical outputs from the notebook when writing the final report.

- Cumulative vaccination increased substantially during the study period.
- Daily vaccination fluctuated, so the 7-day moving average is more useful for identifying sustained peaks.
- First-dose and second-dose activity can be compared separately to understand the progression from initial vaccination to follow-up doses.
- Monthly and weekly aggregation reveals changes in vaccination intensity.
- State-level totals differ considerably; these comparisons should be interpreted alongside state population size because raw dose totals are not population-adjusted.
- Sputnik V, AEFI, and age-specific variables have different reporting periods. Their missing values were deliberately preserved during cleaning.
- Vaccination was already underway during the second COVID-19 wave. Temporal co-movement should not be interpreted as proof of a causal effect.
